Import libraries

In [ ]:
import os
import pandas as pd
import tensorflow as tf

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color'] * 30

Inputs

In [ ]:
# System
atom = "O"
molecule = "O2"
nb_bins = 60
nb_svd_coeff = 8
# Paths
path_to_model = "./../saving/mp_meno/SavedModel/"
path_to_data = "./../../datagen/data/test/"
path_to_figs = "./../postproc/testing/"

Load model

In [ ]:
model = tf.keras.models.load_model(path_to_model)

Get solution

In [ ]:
lbl_mol = [f"n_{molecule}_{i+1}" for i in range(nb_bins)]
lbl_svd = [f"t_p{atom}"] + [f"w{i+1}_pa" for i in range(nb_svd_coeff)]

def get_data(filename):
  sol = pd.read_csv(filename)
  # Extract exact solution
  qoi_t = sol[["t", f"n_{atom}", f"n_{molecule}", "T"]].to_numpy()
  nm_0 = sol[[k+"_0" for k in lbl_mol]].to_numpy()
  f_in = sol[lbl_svd].to_numpy()
  nm_t = sol[lbl_mol].to_numpy()
  # Get predicted solution
  x = [qoi_t, nm_0, f_in]
  nm_t_pred = model(x).numpy().squeeze()
  return qoi_t[:,0], nm_t, nm_t_pred

Plot solution

In [ ]:
def plot_data(
  x=None,
  y=None,
  xref=None,
  yref=None,
  labels=[r'$t$ [s]', r'$X$'],
  scales=['log', 'log'],
  names=None,
  figname=None
):
  # Initialize figures
  fig = plt.figure()
  ax = fig.add_subplot()
  # x axis
  ax.set_xlabel(labels[0])
  ax.set_xscale(scales[0])
  # y axis
  ax.set_ylabel(labels[1])
  ax.set_yscale(scales[1])
  # Plotting
  n, d = y.shape if (x is not None) else yref.shape
  true_style = dict(
    lw=1
  )
  pred_style = dict(
    marker='^',
    fillstyle='none',
    markersize=5,
    markevery=n//20,
    ls='--',
    lw=1
  )
  for i in range(d):
    if (x is not None):
      ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
    if (xref is not None):
      ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
    if (names is not None):
      ax.plot([], [], c=COLORS[i], label=names[i])
  if (names is not None):
    plt.legend()
  if (figname is not None):
    fig.savefig(figname)
  plt.close()

In [ ]:
for (i, case) in enumerate(os.listdir(path_to_data)):
  # Set path for saving
  filename = path_to_data + "/" + case
  path_to_figsi = path_to_figs + "/" + case.split(".")[0]
  os.makedirs(path_to_figsi, exist_ok=True)
  # Get data
  t, nm, nm_pred = get_data(filename)
  # Plot
  plot_data(
    x=t,
    y=nm_pred,
    xref=t,
    yref=nm,
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'linear'],
    figname=path_to_figsi + f"/nm_lin_nocorr.png"
  )
  plot_data(
    x=t,
    y=nm_pred,
    xref=t,
    yref=nm,
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'log'],
    figname=path_to_figsi + f"/nm_log_nocorr.png"
  )